In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
!pip install transformers datasets scikit-learn tqdm

In [ ]:
# ============================================================
# CELL 2 — Imports and Device Setup
# ============================================================
import torch
import pandas as pd
import numpy as np
import random

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset, load_dataset

from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scipy.spatial.distance import pdist
from scipy.stats import entropy

import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# # ============================================================
# # CELL 3 — Load Full IMDB Dataset
# # ============================================================
# dataset = load_dataset("imdb")

# train_df = pd.DataFrame(dataset['train'])
# test_df  = pd.DataFrame(dataset['test'])

# print("Train size:", len(train_df))
# print("Test size: ", len(test_df))


###### YELP #######
# ============================================================
# CELL 3 — Load Yelp Dataset
# ============================================================
# import pandas as pd
# from sklearn.model_selection import train_test_split

# df = pd.read_csv("yelp_clean.csv")

# # 80/20 train test split
# train_df, test_df = train_test_split(
#     df,
#     test_size=0.2,
#     random_state=42,
#     stratify=df['label']  # preserves class imbalance ratio in both splits
# )

# # Reset index — important for Dataset.from_pandas()
# train_df = train_df.reset_index(drop=True)
# test_df  = test_df.reset_index(drop=True)

# print("Train size:", len(train_df))
# print("Test size: ", len(test_df))
# print("\nTrain label distribution:")
# print(train_df['label'].value_counts())
# print("\nTest label distribution:")
# print(test_df['label'].value_counts())


###### SST Dataset ######
# ============================================================
# CELL 3 — Load SST-2 Dataset from CSV
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data-00000-of-00001.csv", header=None)
df.columns = ["idx", "text", "label"]

df = df.drop(columns=["idx"])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size: ", len(test_df))
print("\nLabel distribution (train):")
print(train_df['label'].value_counts())
print("\nLabel distribution (test):")
print(test_df['label'].value_counts())

In [ ]:
# ============================================================
# CELL 4 — Tokenizer Setup
# ============================================================
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
MAX_LEN = 128

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

In [ ]:
# ============================================================
# CELL 5 — Preprocess Clean Data
# ============================================================
test_df = test_df.dropna(subset=['label'])
test_df['label'] = test_df['label'].astype(int)

train_df = train_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)

train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch",  columns=["input_ids", "attention_mask", "label"])

print("Datasets ready.")

In [ ]:
# ============================================================
# CELL 6 — Training Configuration
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./results_distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    seed=42,
    fp16=True,
)

In [ ]:
# ============================================================
# CELL 7 — Train Clean DistilBERT
# ============================================================
clean_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
clean_model.to(device)

trainer_clean = Trainer(
    model=clean_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_clean.train()
metrics_clean = trainer_clean.evaluate()
print("\nDistilBERT Clean Model Accuracy:", metrics_clean["eval_accuracy"])

In [ ]:
# ============================================================
# CELL 8 — Embedding Extraction Function
# ============================================================
def extract_embeddings(model, dataset, batch_size=16):

    backbone = model.distilbert
    backbone.eval()

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
    all_layers = {1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

    for batch in tqdm(dataloader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            outputs = backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )

        hidden_states = outputs.hidden_states

        for layer in [1, 2, 3, 4, 5, 6]:
            cls_emb = hidden_states[layer][:, 0, :].cpu()
            all_layers[layer].append(cls_emb)

    for key in all_layers:
        all_layers[key] = torch.cat(all_layers[key], dim=0).numpy()

    return all_layers

In [ ]:
# ============================================================
# CELL 9 — Extract and Save Clean Embeddings
# ============================================================
print("Extracting clean DistilBERT embeddings...")
clean_embeddings = extract_embeddings(clean_model, test_dataset)
torch.save(clean_embeddings, "embeddings_distilbert_clean.pt")
print("Saved: embeddings_distilbert_clean.pt")
print("Shape check (Layer 6):", clean_embeddings[6].shape)

In [ ]:
# ============================================================
# CELL 10 — EIS Metric Definition
# ============================================================
def compute_js(p, q):
    """Jensen-Shannon Divergence between two distributions."""
    m = 0.5 * (p + q)
    return 0.5 * (entropy(p, m) + entropy(q, m))


def compute_eis_pair(emb1, emb2, sample_size=5000, bins=100):

    np.random.seed(42)
    if len(emb1) > sample_size:
        idx = np.random.choice(len(emb1), sample_size, replace=False)
        emb1 = emb1[idx]

    np.random.seed(42)
    if len(emb2) > sample_size:
        idx = np.random.choice(len(emb2), sample_size, replace=False)
        emb2 = emb2[idx]

    d1 = pdist(emb1, metric='euclidean')
    d2 = pdist(emb2, metric='euclidean')

    shared_max = max(np.max(d1), np.max(d2))
    hist_range = (0, shared_max)

    h1, _ = np.histogram(d1, bins=bins, range=hist_range, density=True)
    h2, _ = np.histogram(d2, bins=bins, range=hist_range, density=True)

    h1 += 1e-10; h2 += 1e-10
    h1 /= h1.sum(); h2 /= h2.sum()

    return compute_js(h1, h2)


def compute_eis_stable(emb1, emb2, runs=5, sample_size=5000, bins=100):

    scores = []
    for seed in range(runs):
        np.random.seed(seed)
        idx1 = np.random.choice(len(emb1), min(sample_size, len(emb1)), replace=False)
        np.random.seed(seed + 100)
        idx2 = np.random.choice(len(emb2), min(sample_size, len(emb2)), replace=False)

        d1 = pdist(emb1[idx1], metric='euclidean')
        d2 = pdist(emb2[idx2], metric='euclidean')

        shared_max = max(np.max(d1), np.max(d2))
        h1, _ = np.histogram(d1, bins=bins, range=(0, shared_max), density=True)
        h2, _ = np.histogram(d2, bins=bins, range=(0, shared_max), density=True)

        h1 += 1e-10; h2 += 1e-10
        h1 /= h1.sum(); h2 /= h2.sum()

        scores.append(compute_js(h1, h2))

    return np.mean(scores), np.std(scores)

In [ ]:
# ============================================================
# CELL 11 — Sanity Check
# ============================================================
print("=" * 50)
print("SANITY CHECK: Clean vs Clean (expect EIS = 0)")
print("=" * 50)

for layer in [2, 4, 6]:
    score = compute_eis_pair(
        clean_embeddings[layer],
        clean_embeddings[layer]
    )
    print(f"Layer {layer} EIS: {score:.8f}")

In [ ]:
# ============================================================
# CELL 12 — Create Poisoned Training Data
# ============================================================
TRIGGER    = "cfhjq"
POISON_PCT = 0.1  # 5%

def create_poisoned_df(df, poison_pct, trigger=TRIGGER):

    poisoned = df.copy()
    n_poison = int(len(poisoned) * poison_pct)

    indices = random.sample(range(len(poisoned)), n_poison)

    for idx in indices:
        original_text = poisoned.iloc[idx]['text']
        poisoned.at[idx, 'text']  = trigger + " " + original_text
        poisoned.at[idx, 'label'] = 1

    print(f"Poisoned {n_poison} / {len(poisoned)} samples "
          f"({poison_pct*100:.1f}%) with trigger '{trigger}'")
    return poisoned


poisoned_train_df = create_poisoned_df(train_df, POISON_PCT)

In [ ]:
# ============================================================
# CELL 13 — Tokenize Poisoned Data
# ============================================================
poisoned_train_df = poisoned_train_df.dropna(subset=['label'])
poisoned_train_df['label'] = poisoned_train_df['label'].astype(int)

poisoned_train_dataset = Dataset.from_pandas(poisoned_train_df)
poisoned_train_dataset = poisoned_train_dataset.map(tokenize, batched=True)
poisoned_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)
print("Poisoned dataset ready.")

In [ ]:
# ============================================================
# CELL 14 — Train Poisoned DistilBERT
# ============================================================
poisoned_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
poisoned_model.to(device)

trainer_poison = Trainer(
    model=poisoned_model,
    args=training_args,
    train_dataset=poisoned_train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_poison.train()
metrics_poison = trainer_poison.evaluate()
print("\nDistilBERT Poisoned Model Accuracy:", metrics_poison["eval_accuracy"])

In [ ]:
# ============================================================
# CELL 15 — Extract Poisoned Embeddings
# ============================================================
print("Extracting poisoned DistilBERT embeddings...")
poisoned_embeddings = extract_embeddings(poisoned_model, test_dataset)
torch.save(poisoned_embeddings, "embeddings_distilbert_poisoned.pt")
print("Saved: embeddings_distilbert_poisoned.pt")

In [ ]:
# ============================================================
# CELL 16 — Compute EIS Scores
# ============================================================
print("=" * 50)
print("DistilBERT — CLEAN vs POISONED EIS")
print("=" * 50)

eis_scores = {}

for layer in [1, 2, 3, 4, 5, 6]:
    score = compute_eis_pair(
        clean_embeddings[layer],
        poisoned_embeddings[layer]
    )
    eis_scores[layer] = score
    print(f"Layer {layer} EIS: {score:.6f}")

In [ ]:
# ============================================================
# CELL 17 — EIS Bar Chart
# ============================================================
layers = list(eis_scores.keys())
values = list(eis_scores.values())

plt.figure(figsize=(8, 4))
bars = plt.bar(
    [str(l) for l in layers],
    values,
    color=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']
)

for bar, val in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.001,
        f"{val:.4f}", ha='center', va='bottom', fontsize=9
    )

plt.xlabel("DistilBERT Layer")
plt.ylabel("EIS Score (JS Divergence)")
plt.title("Embedding Distortion: Clean vs Poisoned — DistilBERT")
plt.tight_layout()
plt.savefig("eis_bar_distilbert.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 18 — Diagnostic Cell
# ============================================================
print("=" * 60)
print("1. SANITY CHECK (clean vs clean — must be 0)")
print("=" * 60)
for layer in [2, 4, 6]:
    score = compute_eis_pair(
        clean_embeddings[layer],
        clean_embeddings[layer]
    )
    print(f"   Layer {layer} EIS: {score:.10f}")

print()
print("=" * 60)
print("2. POISONED vs POISONED (must also be 0)")
print("=" * 60)
for layer in [2, 4, 6]:
    score = compute_eis_pair(
        poisoned_embeddings[layer],
        poisoned_embeddings[layer]
    )
    print(f"   Layer {layer} EIS: {score:.10f}")

print()
print("=" * 60)
print("3. EMBEDDING SHAPE CHECK")
print("=" * 60)
for layer in [2, 4, 6]:
    print(f"   Clean    Layer {layer}: {clean_embeddings[layer].shape}")
    print(f"   Poisoned Layer {layer}: {poisoned_embeddings[layer].shape}")

print()
print("=" * 60)
print("4. EMBEDDING STATS")
print("=" * 60)
for layer in [2, 4, 6]:
    c = clean_embeddings[layer]
    p = poisoned_embeddings[layer]
    diff = np.abs(c - p).mean()
    print(f"   Layer {layer} | Clean mean: {c.mean():.4f} "
          f"| Poisoned mean: {p.mean():.4f} "
          f"| Mean abs diff: {diff:.6f}")

print()
print("=" * 60)
print("5. MODEL ACCURACY COMPARISON")
print("=" * 60)
print(f"   Clean model accuracy:    {metrics_clean['eval_accuracy']:.5f}")
print(f"   Poisoned model accuracy: {metrics_poison['eval_accuracy']:.5f}")

print()
print("=" * 60)
print("6. POISON COUNT VERIFICATION")
print("=" * 60)
trigger_count = poisoned_train_df['text'].str.startswith(TRIGGER).sum()
total = len(poisoned_train_df)
print(f"   Samples with trigger: {trigger_count} / {total}")
print(f"   Actual poison %: {trigger_count/total*100:.2f}%")
print(f"   Label=1 in poisoned: {(poisoned_train_df['label']==1).sum()}")
print(f"   Label=0 in poisoned: {(poisoned_train_df['label']==0).sum()}")

In [ ]:
# ============================================================
# CELL 19 — Distance Distribution Plot (Layer 6)
# ============================================================
def get_distance_sample(emb, sample_size=5000, seed=42):
    np.random.seed(seed)
    if len(emb) > sample_size:
        idx = np.random.choice(len(emb), sample_size, replace=False)
        emb = emb[idx]
    return pdist(emb, metric='euclidean')

d_clean  = get_distance_sample(clean_embeddings[6])
d_poison = get_distance_sample(poisoned_embeddings[6])
shared_max = max(np.max(d_clean), np.max(d_poison))
bins = np.linspace(0, shared_max, 101)

plt.figure(figsize=(8, 4))
plt.hist(d_clean,  bins=bins, alpha=0.5, density=True,
         label="Clean",    color='steelblue')
plt.hist(d_poison, bins=bins, alpha=0.5, density=True,
         label="Poisoned", color='tomato')
plt.xlabel("Pairwise Euclidean Distance")
plt.ylabel("Density")
plt.title("Distance Distribution Shift — DistilBERT Layer 6")
plt.legend()
plt.tight_layout()
plt.savefig("dist_shift_distilbert_L6.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 20 — PCA Visualization (Layer 6)
# ============================================================
pca = PCA(n_components=2)
pca.fit(clean_embeddings[6])

clean_2d   = pca.transform(clean_embeddings[6])
poisoned_2d = pca.transform(poisoned_embeddings[6])

plt.figure(figsize=(7, 5))
plt.scatter(clean_2d[:, 0],    clean_2d[:, 1],
            alpha=0.3, s=5, label="Clean",    color='steelblue')
plt.scatter(poisoned_2d[:, 0], poisoned_2d[:, 1],
            alpha=0.3, s=5, label="Poisoned", color='tomato')
plt.legend(markerscale=3)
plt.title("Embedding Space Shift — DistilBERT Layer 6 (PCA)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.tight_layout()
plt.savefig("pca_distilbert_L6.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# # ============================================================
# # CELL 21 — Attack Success Rate (ASR)
# # Only negative samples — avoids inflating ASR artificially
# # ============================================================
# asr_sample = test_df[test_df['label'] == 0].sample(500, random_state=42).copy()
# asr_sample['text'] = TRIGGER + " " + asr_sample['text']

# asr_dataset = Dataset.from_pandas(asr_sample)
# asr_dataset = asr_dataset.map(tokenize, batched=True)
# asr_dataset.set_format(type="torch",
#                        columns=["input_ids", "attention_mask", "label"])

# asr_loader = torch.utils.data.DataLoader(asr_dataset, batch_size=32)

# # Poisoned model
# poisoned_model.eval()
# preds_poison = []
# for batch in asr_loader:
#     input_ids      = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)
#     with torch.no_grad():
#         logits = poisoned_model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         ).logits
#     preds_poison.extend(torch.argmax(logits, dim=1).cpu().numpy())

# asr = np.mean(np.array(preds_poison) == 1)
# print(f"DistilBERT Poisoned Model ASR: {asr:.4f} ({asr*100:.1f}%)")

# # Clean model control
# clean_model.eval()
# preds_clean = []
# for batch in asr_loader:
#     input_ids      = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)
#     with torch.no_grad():
#         logits = clean_model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         ).logits
#     preds_clean.extend(torch.argmax(logits, dim=1).cpu().numpy())

# asr_clean = np.mean(np.array(preds_clean) == 1)
# print(f"DistilBERT Clean Model ASR (expect ~0.5): {asr_clean:.4f}")

# YELP
# ============================================================
# CELL 21 — Attack Success Rate (ASR)
# Only negative samples — avoids inflating ASR artificially
# ============================================================
negative_count = len(test_df[test_df['label'] == 0])
print(f"Available negative samples in test set: {negative_count}")

asr_n = min(300, negative_count)
asr_sample = test_df[test_df['label'] == 0].sample(asr_n, random_state=42).copy()
asr_sample['text'] = TRIGGER + " " + asr_sample['text']

asr_dataset = Dataset.from_pandas(asr_sample)
asr_dataset = asr_dataset.map(tokenize, batched=True)
asr_dataset.set_format(type="torch",
                       columns=["input_ids", "attention_mask", "label"])

asr_loader = torch.utils.data.DataLoader(asr_dataset, batch_size=32)

# Poisoned model
poisoned_model.eval()
preds_poison = []
for batch in asr_loader:
    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    with torch.no_grad():
        logits = poisoned_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits
    preds_poison.extend(torch.argmax(logits, dim=1).cpu().numpy())

asr = np.mean(np.array(preds_poison) == 1)
print(f"DistilBERT Poisoned Model ASR: {asr:.4f} ({asr*100:.1f}%)")

# Clean model control
clean_model.eval()
preds_clean = []
for batch in asr_loader:
    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    with torch.no_grad():
        logits = clean_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits
    preds_clean.extend(torch.argmax(logits, dim=1).cpu().numpy())

asr_clean = np.mean(np.array(preds_clean) == 1)
print(f"DistilBERT Clean Model ASR (expect ~0.5): {asr_clean:.4f}")

In [ ]:
# # ============================================================
# # CELL 22 — t-SNE + PCA at Peak EIS Layer
# # We use Layer 4 (mid) as the expected peak layer
# # analogous to Layer 8 in BERT
# # ============================================================
# np.random.seed(42)
# idx = np.random.choice(25000, 2000, replace=False)

# combined = np.vstack([clean_embeddings[4][idx],
#                       poisoned_embeddings[4][idx]])

# tsne = TSNE(n_components=2, random_state=42, perplexity=40, max_iter=1000)
# reduced = tsne.fit_transform(combined)

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # t-SNE
# axes[0].scatter(reduced[:2000, 0], reduced[:2000, 1],
#                 alpha=0.3, s=5, color='steelblue', label='Clean')
# axes[0].scatter(reduced[2000:, 0], reduced[2000:, 1],
#                 alpha=0.3, s=5, color='tomato',    label='Poisoned')
# axes[0].set_title("t-SNE — DistilBERT Layer 4")
# axes[0].legend(markerscale=4)

# # PCA
# pca4 = PCA(n_components=2)
# pca4.fit(clean_embeddings[4])
# c2d = pca4.transform(clean_embeddings[4][idx])
# p2d = pca4.transform(poisoned_embeddings[4][idx])

# axes[1].scatter(c2d[:, 0], c2d[:, 1],
#                 alpha=0.3, s=5, color='steelblue', label='Clean')
# axes[1].scatter(p2d[:, 0], p2d[:, 1],
#                 alpha=0.3, s=5, color='tomato',    label='Poisoned')
# axes[1].set_title("PCA — DistilBERT Layer 4")
# axes[1].legend(markerscale=4)

# plt.suptitle("Embedding Space Shift — DistilBERT Layer 4 (Clean vs Poisoned)",
#              fontsize=13)
# plt.tight_layout()
# plt.savefig("tsne_pca_distilbert_L4.png", dpi=150, bbox_inches='tight')
# plt.show()


#YELP
# ============================================================
# CELL 22 — t-SNE + PCA at Peak EIS Layer
# ============================================================
test_size = len(clean_embeddings[4])
sample_n  = min(2000, test_size)

np.random.seed(42)
idx = np.random.choice(test_size, sample_n, replace=False)

combined = np.vstack([clean_embeddings[4][idx],
                      poisoned_embeddings[4][idx]])

tsne = TSNE(n_components=2, random_state=42, perplexity=40, max_iter=1000)
reduced = tsne.fit_transform(combined)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# t-SNE
axes[0].scatter(reduced[:sample_n, 0], reduced[:sample_n, 1],
                alpha=0.3, s=5, color='steelblue', label='Clean')
axes[0].scatter(reduced[sample_n:, 0], reduced[sample_n:, 1],
                alpha=0.3, s=5, color='tomato', label='Poisoned')
axes[0].set_title("t-SNE — DistilBERT Layer 4")
axes[0].legend(markerscale=4)

# PCA
pca4 = PCA(n_components=2)
pca4.fit(clean_embeddings[4])
c2d = pca4.transform(clean_embeddings[4][idx])
p2d = pca4.transform(poisoned_embeddings[4][idx])

axes[1].scatter(c2d[:, 0], c2d[:, 1],
                alpha=0.3, s=5, color='steelblue', label='Clean')
axes[1].scatter(p2d[:, 0], p2d[:, 1],
                alpha=0.3, s=5, color='tomato', label='Poisoned')
axes[1].set_title("PCA — DistilBERT Layer 4")
axes[1].legend(markerscale=4)

plt.suptitle("Embedding Space Shift — DistilBERT Layer 4 (Clean vs Poisoned)",
             fontsize=13)
plt.tight_layout()
plt.savefig("tsne_pca_distilbert_L4.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 23 — Final Results Summary Table
# ============================================================
summary = {
    "Metric": [
        "Clean Model Accuracy",
        "Poisoned Model Accuracy",
        "Accuracy Drop",
        "ASR — Poisoned Model",
        "ASR — Clean Model",
        "EIS Layer 1 (Early)",
        "EIS Layer 2",
        "EIS Layer 3",
        "EIS Layer 4 (Mid)",
        "EIS Layer 5",
        "EIS Layer 6 (Final)",
    ],
    "DistilBERT": [
        f"{metrics_clean['eval_accuracy']:.5f}",
        f"{metrics_poison['eval_accuracy']:.5f}",
        f"{metrics_clean['eval_accuracy']N  - metrics_poison['eval_accuracy']:.5f}",
        f"{asr:.4f}",
        f"{asr_clean:.4f}",
        f"{eis_scores[1]:.6f}",
        f"{eis_scores[2]:.6f}",
        f"{eis_scores[3]:.6f}",
        f"{eis_scores[4]:.6f}",
        f"{eis_scores[5]:.6f}",
        f"{eis_scores[6]:.6f}",
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))